# CTB ProSiT models: minimal loader

This notebook loads the common Inductive-Miner Petri net (PNML) and one ProSiT parameter file (JSON), using the save/load pattern documented by ProSiT. Select a model below, then run all cells.

In [ ]:
%pip install -q "prosit-pm==1.0.3" typing_extensions


In [ ]:
from pathlib import Path
import pm4py
from prosit import SimulatorParameters, SimulatorEngine
from IPython.display import display

# In a local Jupyter session, leave this as Path.cwd().
# In Google Colab after mounting Drive, set ROOT to the uploaded folder, e.g.:
# ROOT = Path('/content/drive/MyDrive/CTB_ProSiT_minimal_handover')
ROOT = Path.cwd()
if not (ROOT / 'models').is_dir():
    raise FileNotFoundError('Open the notebook from the uploaded handover folder, or set ROOT to that folder.')

MODEL = 'baseline'  # choose: baseline, t22_closed, demand_plus_20pct
PNML_FILE = ROOT / 'models' / 'ctb_inductive_miner.pnml'
JSON_FILE = ROOT / 'models' / f'params_{MODEL}.json'

net, initial_marking, final_marking = pm4py.read_pnml(str(PNML_FILE))
params = SimulatorParameters(net, initial_marking, final_marking)
params.from_json(str(JSON_FILE))
simulator = SimulatorEngine(params)

print(f'Loaded: {MODEL}')
print(f'Petri net: {len(net.places)} places, {len(net.transitions)} transitions, {len(net.arcs)} arcs')
print(f'Resources: {len(params.resources)}; activities: {len(params.act_to_resources)}')
print(f'Rules mode: {params.rules_mode}; workload features: {params.use_workload_features}')

In [ ]:
for name in ('baseline', 't22_closed', 'demand_plus_20pct'):
    other = SimulatorParameters(net, initial_marking, final_marking)
    other.from_json(str(ROOT / 'models' / f'params_{name}.json'))
    rmg = sorted({r for a in ('RMG_receive', 'RMG_delivery', 'RMG_mixed') for r in other.act_to_resources[a]})
    arrival_mean = other.arrival_time_distribution.rules[0]['value']
    print(f'{name:18} RMG blocks={len(rmg):2}; T22 eligible={"T22" in rmg}; mean inter-arrival={arrival_mean:.4f} min')

In [ ]:
# Optional ProSiT simulation demonstration. This is deliberately small.
# It demonstrates that the loaded model is executable; it is NOT the thesis replication.
RUN_DEMONSTRATION = False
if RUN_DEMONSTRATION:
    simulated_log = simulator.apply(n_traces=5)
    display(simulated_log.head())

## Why both PNML and JSON?

PNML preserves the **control-flow backbone**: places, transitions, arcs, and the initial/final markings. ProSiT JSON stores the learned **simulation parameters** attached to that backbone: routing and resource rules, calendars, capacities, execution and waiting-time rules, arrivals, and data attributes. Neither file alone is sufficient to reconstruct a ProSiT simulation model.

These JSON files are portable, human-readable inspection files. ProSiT's JSON export omits cached stochastic `sampled` values. In addition, CTB has missing values in empirical attribute tuples; the files use JSON `null`/Python `None` so that ProSiT's standard `from_json()` can load them. Consequently, this notebook is the transparent model-inspection handover. Exact numerical replication of the archived thesis tables requires the frozen binary parameter bundles and the controlled runner, because it must preserve those cached runtime samples as well.